# Arm L — Molmo2-O-7B (Stage 4) + Qwen3-VL-8B/adapters (Stage 13, Stage 12) — mixed-model full-stack E2E harness (GPU, Colab-only)

Local-model counterpart to Arm P v3 (`src/e2e_harness/poc_run_arm_p_v3.py`, GPT-5.5-low),
extending this project's Qwen-only Arm L (`ArmL_QwenVL_FullStack_GPUOnly.ipynb`) with the
REAL, unmodified stage 13 (entity validation) and stage 12 (relation validation) drivers,
using a DELIBERATELY MIXED per-stage model assignment (user-confirmed mapping):

| Stage | Model | Why |
|---|---|---|
| 4 — Symbol detection | **Molmo2-O-7B**, base, zero-shot | Native pixel-pointing, this project's strongest localization candidate (CLAUDE.md candidate table) |
| 13 — Entity validation | **Qwen3-VL-8B-Instruct + `v3-stage13` LoRA** | Domain-adapted, task-specific adapter already trained for this exact validation task |
| 12 — Relation validation | **Qwen3-VL-8B-Instruct + `v3-relation` LoRA** | Same base, different adapter — loaded together via `peft` multi-adapter, not two model copies |

```
real system prompt (none needed — Molmo has no tool-calling and no JSON schema for stage 4)
  -> PaddleOCR (stage 1.5 sub) -> real tile grid (1024/205, SAME grid Arm P/Qwen-only Arm L use)
  -> Molmo2 per-tile, per-entity-type pointing calls -> nearest-OCR-word pairing (see §8 — load-bearing)
  -> real NMS/compose -> real stage_06_run (line tracing)
  -> real detections_to_entities / build_relations
  -> score checkpoint 1 (pre-13/12) vs PID2Graph ground truth (contracted equipment edges)
  -> FREE Molmo, load Qwen3-VL-8B + BOTH adapters (multi-adapter, one base model)
  -> real stage_13_run (vlm_runner = LocalQwenRunner, adapter="v3-stage13")
  -> score checkpoint 2 (post-13)
  -> real stage_12_run (vlm_runner = LocalQwenRunner, adapter="v3-relation")
  -> score checkpoint 3 (post-12)
```

**This CANNOT run locally** — Colab-only, same GPU/Drive rules as every other notebook in
this project (no Google Drive anywhere; all shared storage — model/adapter weights,
datasets, this project's own private code — goes through Hugging Face).

## What is proven vs. what is a first attempt (read this before trusting any number)

**Proven, reused verbatim from working code:**
- Molmo2-O-7B load recipe (`AutoModelForImageTextToText`/`AutoProcessor`,
  `trust_remote_code=True`, `dtype="auto"`, `device_map="cuda"`) — from
  `Stage105_SkidMatrix_Molmo2_Qwen_Adapters_GPUOnly.ipynb` / `Stage4_Phase4_MolmoZeroShot.ipynb`.
- Molmo2 `<points coords="...">` parsing — `e2e_bench/backends/parse_molmo.py`, ported
  verbatim from the recorded Stage 4 zero-shot run (F1=0.628 at **tile=512, upscale=2**).
- `peft` multi-adapter pattern (`PeftModel.from_pretrained(base, path1, adapter_name=...)`
  then `.load_adapter(path2, adapter_name=...)` then `.set_adapter(name)`) — verified
  against `Stage105_SkidMatrix_Molmo2_Qwen_Adapters_GPUOnly.ipynb` section 6, reused exactly.
- Real `stage_13_run`/`stage_12_run` injectable-`vlm_runner` pattern — proven end-to-end by
  Arm P v3 with `RealOpenAIRunner` (OpenAI backend); this notebook's
  `LocalQwenMessagesClient`/`LocalQwenRunner` (`src/e2e_bench/assembly/local_qwen_client.py`)
  is the same shim retargeted at a local Qwen call, unit-tested locally (see file docstring)
  against synthetic Anthropic-shaped inputs covering: fenced-JSON success, plain-text
  keep/remove fallback, plain-text yes/no fallback, an arbitrary 3-image content list
  (stage 12's shape), and total-garbage-output -> empty payload. **Never run against the
  real model** — Colab-only, not executable in this environment.
- `contract_to_equipment_edges` (degree-4 heuristic, `ground_truth.py`) used for relation
  scoring at ALL THREE checkpoints, not PID2Graph's raw routing edges — per the load-bearing
  2026-07-16 discovery that raw edges contain ZERO direct equipment&lt;-&gt;equipment pairs,
  which forces relation-F1=0.0 everywhere regardless of model quality if left uncontracted.

## Decisions made in this build — flagged, not silently assumed

**(1) Molmo2 has no native per-point class label — 6 separate "point to every X" calls per
tile, one per detectable benchmark entity_type, instead of 1 generic call.** Necessary
because — unlike Gupta's class-agnostic Stage 4 metric elsewhere in this project —
`e2e_harness/graph_matcher.py`'s `match_entities` here is **type-sensitive**: a predicted
entity only matches a GT entity if `AGENT_TO_GT_LABEL[entity_type]` equals the GT node's
label. Tagging every Molmo detection with one fallback type (e.g. `"general"`) would only
ever match GT nodes literally labeled `"general"`, artificially collapsing recall against
valve/instrumentation/pump/tank/inlet_outlet ground truth — the same "everyone merged"
failure mode already documented in `Stage105_SkidMatrix...ipynb`'s intro. The tradeoff:
**6x the Molmo calls per tile** (one per type: valve, instrumentation, pump, tank,
inlet_outlet, general — `"asset"` excluded, see decision 3) and **no cross-type dedup** —
if two different type-prompts each point near the same real symbol, both survive as
separate detections (NMS in `nms.py` only dedups WITHIN the same entity_type across tiles,
never across types). This inflates false positives; a real fix (single call with
per-point type tags in the pointing format) is untested and out of scope for this pass.

**(2) Real 1024px/205px-overlap prod tiling grid used for Stage 4, NOT Molmo's own tuned
512px/upscale=2 zero-shot config.** The recorded F1=0.628 was measured at tile=512. This
notebook applies the SAME upscale=2 factor to the real 1024px tiles (2048px effective model
input) as the closest honest adaptation — **this exact combination (1024 grid + 2x upscale)
is untested**; if Stage 4 detection quality looks unexpectedly poor, this is the first thing
to revisit (try upscale=1, or a genuinely re-tuned upscale factor for 1024 tiles).

**(3) `"asset"` (the benchmark ontology's skid-membership umbrella type) is excluded from
Molmo's per-type prompting** — it has no real drawn icon (`graph_matcher.py`'s own comment:
`"asset": "general"  # no direct GT equivalent ... benchmark-only umbrella type`) and Stage 4
detection is about drawn symbols, not derived groupings.

**(4) OCR-word pairing radius is an untuned guess (120px, tile-local, pre-upscale
coordinates), never validated against real data in this pass** — see §8 for the full
rationale and the exact reason this pairing step exists at all (a hard, code-verified
requirement, not a nice-to-have).

**(5) Whether `LocalQwenMessagesClient`'s fenced-JSON-first / plain-text-fallback design
actually produces PARSEABLE answers from the real `v3-stage13`/`v3-relation` adapters under
LoRA is COMPLETELY UNTESTED** — same honesty standard the Qwen-only Arm L notebook already
sets for its own stage-4 JSON-in-prompt parseability. `parse_json_common.py`'s own
docstrings record that these two adapters were actually TRAINED on plain "keep"/"remove" and
"yes"/"no" text, not JSON — §10 below explains why this client tries fenced-JSON FIRST
anyway (matching the task's specified technique and the driver's real declared tool schema)
with the plain-text convention only as a fallback, and flags this tension explicitly as a
real, unresolved risk to watch in the first real run's parse-failure counts.


## 1. Config

In [ ]:
# ── Model / adapter config ──────────────────────────────────────────────────
MOLMO_MODEL_ID = "allenai/Molmo2-O-7B"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"        # base-model LoRA adapters live here
ADAPTERS = {                                            # adapter_name (peft) -> path in CKPT_REPO
    "stage13": "v3-stage13/latest",
    "relation": "v3-relation/latest",
}

# ── HF (datasets, private agent-code zip). No Google Drive, ever. ──────────
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"
DATA_REPO = "timthy45/pnid-extraction-datasets"       # public-within-org dataset repo, holds PID2Graph.zip etc
AGENT_SRC_REPO = "timthy45/pnid-agent-src"             # PRIVATE dataset repo — scripts/package_agent_src_for_colab.sh
AGENT_SRC_FILE = "agent_src/latest.zip"                # stable pointer the packaging script always overwrites

# ── Stage 4 (Molmo2) config — see intro decisions (1)/(2)/(4) ──────────────
MOLMO_TILE_UPSCALE = 2.0     # applied to the real 1024px grid tiles — UNTESTED at this tile size (decision 2)
OCR_PAIR_RADIUS_PX = 120     # tile-local, PRE-upscale pixels — untuned guess (decision 4), try 80-150
MOLMO_MAX_NEW_TOKENS = 600

# ── Stage 13/12 (Qwen + adapters) config ────────────────────────────────────
QWEN_MAX_NEW_TOKENS = 1024

# ── Holdout sheet ────────────────────────────────────────────────────────────
# One of the 4 frozen sheets in e2e_holdout_ids.json. Same primary sheet Arm P v2/v3 and
# the Qwen-only Arm L notebook use, for direct comparability.
HOLDOUT_TREE = "PID2Graph OPEN100"
HOLDOUT_STEM = "8"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "PASTE_YOUR_HF_TOKEN_HERE", "paste your HF token"


## 2. Install (GPU-runtime prep)

`transformers==4.57.1` pinned — required by Molmo2's remote-code processor AND known-good
for `Qwen3-VL-8B-Instruct` + `peft` (same pin as every other Qwen/Molmo notebook in this
project). `libmagic1` via apt — the one system-level dependency `pnid_agent`'s Stage 0
ingestion code needs transitively (`python-magic`).

**Deliberately NOT installing `torch` generically** — Colab's preinstalled CUDA build gets
silently replaced by a CPU-only PyPI wheel if you do (bit this project's Molmo notebook
twice already, per that notebook's own install-section note).

In [ ]:
!apt-get -qq update && apt-get -qq install -y libmagic1 > /dev/null
!pip install -q transformers==4.57.1 accelerate peft huggingface_hub
!pip install -q paddleocr paddlepaddle
!pip install -q pymupdf python-magic

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


## 3. Private code: pnid_agent + its monorepo deps + this repo's own e2e_bench/e2e_harness

Identical to the Qwen-only Arm L notebook's section 3 — downloads the zip
`scripts/package_agent_src_for_colab.sh` builds and pushes, installs the 4 private editable
packages in dependency order, then puts `e2e_bench`/`e2e_harness` on `sys.path`.

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

AGENT_SRC_ROOT = Path("/content/agent_src")
AGENT_SRC_ROOT.mkdir(exist_ok=True)

zp = hf_hub_download(repo_id=AGENT_SRC_REPO, filename=AGENT_SRC_FILE,
                      repo_type="dataset", token=HF_TOKEN)
with zipfile.ZipFile(zp) as zf:
    zf.extractall(AGENT_SRC_ROOT)
print("extracted:", sorted(p.name for p in AGENT_SRC_ROOT.iterdir()))


In [ ]:
# Install order matters: rive_security -> entity_operations -> rive_adk -> pnid_agent.
!pip install -q -e /content/agent_src/shared/security
!pip install -q -e /content/agent_src/shared/entity_operations
!pip install -q -e /content/agent_src/shared/rive_adk
!pip install -q -e "/content/agent_src/agents/pnid-intelligence-agent"

import sys
sys.path.insert(0, "/content/agent_src/pid_ml_src")
print("sys.path updated for e2e_bench / e2e_harness")


In [ ]:
# Fail fast with a clear message rather than a confusing import error later.
import importlib

REQUIRED_MODULES = [
    "pnid_agent.models.page_classification", "pnid_agent.models.page_ocr",
    "pnid_agent.models.detections", "pnid_agent.models.line_tracing",
    "pnid_agent.models.drawing_document", "pnid_agent.models.rive_ontology",
    "pnid_agent.shared.coord_ops",
    "pnid_agent.stages.tile_segmentation.grid",
    "pnid_agent.sub_agents.symbol_detection.nms",
    "pnid_agent.sub_agents.symbol_detection.driver",
    "pnid_agent.sub_agents.symbol_detection.tile_words",
    "pnid_agent.stages.line_tracing.driver",
    "pnid_agent.stages.graph_construction.relations",
    "pnid_agent.sub_agents.entity_validation.driver",
    "pnid_agent.sub_agents.relation_validation.driver",
    "e2e_bench.assembly.document", "e2e_bench.assembly.entities",
    "e2e_bench.assembly.local_qwen_client",
    "e2e_bench.backends.parse_molmo", "e2e_bench.backends.parse_json_common",
    "e2e_bench.converters.stage01_classification", "e2e_bench.converters.stage04_detection",
    "e2e_bench.ontology", "e2e_bench.types",
    "e2e_harness.graph_matcher", "e2e_harness.ground_truth", "e2e_harness.holdout",
]
missing = []
for mod in REQUIRED_MODULES:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing.append(f"{mod}: {type(e).__name__}: {e}")
if missing:
    raise RuntimeError("Missing/broken imports:\n  " + "\n  ".join(missing))
print(f"all {len(REQUIRED_MODULES)} required modules import cleanly")


## 4. Ground-truth data — PID2Graph

Identical to the Qwen-only Arm L notebook's section 4: downloads `pid2graph/PID2Graph.zip`
from `DATA_REPO`, extracts it, monkeypatches `e2e_harness/holdout.py`'s Mac-local hardcoded
path to this Colab session's extraction path.

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data"); DATA.mkdir(exist_ok=True)
zp = hf_hub_download(repo_id=DATA_REPO, filename="pid2graph/PID2Graph.zip",
                      repo_type="dataset", token=HF_TOKEN)
pid2graph_dir = DATA / "pid2graph"
if not (pid2graph_dir / "PID2Graph" / "Complete").exists():
    pid2graph_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(pid2graph_dir)

COMPLETE_ROOT = str(pid2graph_dir / "PID2Graph" / "Complete")
assert Path(COMPLETE_ROOT).exists(), COMPLETE_ROOT
print("PID2Graph Complete tree ready at", COMPLETE_ROOT)


In [ ]:
import e2e_harness.holdout as holdout_mod

holdout_mod._PID2GRAPH_COMPLETE_ROOT = COMPLETE_ROOT

def resolve_sheet(tree: str, stem: str) -> dict:
    base = f"{COMPLETE_ROOT}/{tree}/{stem}"
    sheet_id = f"{tree.replace(' ', '')}_{stem}"
    return {"sheet_id": sheet_id, "graphml_path": f"{base}.graphml", "png_path": f"{base}.png"}

sheet = resolve_sheet(HOLDOUT_TREE, HOLDOUT_STEM)
for k, v in sheet.items():
    if k != "sheet_id":
        assert Path(v).exists(), v
print(sheet)


## 5. Load Molmo2-O-7B (Stage 4 model)

Verbatim load recipe from `Stage105_SkidMatrix_Molmo2_Qwen_Adapters_GPUOnly.ipynb` /
`Stage4_Phase4_MolmoZeroShot.ipynb`: `trust_remote_code=True`, `dtype="auto"`,
`device_map="cuda"`. This is the moment cascade mode begins (H6, same rule the Qwen-only
Arm L notebook documents) — Molmo stays resident in GPU memory through all of Stage 4's
per-tile, per-type calls, then is explicitly freed (§11) before Qwen loads for stages 13/12.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

molmo_processor = AutoProcessor.from_pretrained(MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto")
molmo_model = AutoModelForImageTextToText.from_pretrained(
    MOLMO_MODEL_ID, trust_remote_code=True, dtype="auto", device_map="cuda")
print("Molmo2-O-7B loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def molmo_generate(image, prompt_text, max_new_tokens=MOLMO_MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": [
        {"type": "text", "text": prompt_text}, {"type": "image", "image": image}]}]
    inputs = molmo_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(molmo_model.device)
    with torch.no_grad():
        out = molmo_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return molmo_processor.batch_decode(gen, skip_special_tokens=True)[0].strip()


## 6. Per-entity-type Molmo prompts (decision 1 — why 6 calls per tile, not 1)

`e2e_bench.ontology.entity_types()` is the real benchmark ontology's 7 types
(`valve, instrumentation, pump, tank, general, inlet_outlet, asset`). `"asset"` is excluded
(decision 3 — no drawn icon). Each of the other 6 gets its own short, Molmo-native
"point to X" prompt (same short-imperative style `Stage4_Phase4_MolmoZeroShot.ipynb`'s v3
prompt already validated works better than long paragraph prompts for this model).

In [ ]:
from e2e_bench.ontology import entity_type_names, entity_types

_ALL_TYPES = entity_types()
assert set(_ALL_TYPES) == {"valve", "instrumentation", "pump", "tank", "general", "inlet_outlet", "asset"}
MOLMO_ENTITY_TYPES = [t for t in _ALL_TYPES if t != "asset"]  # decision 3

MOLMO_PROMPTS_BY_TYPE = {
    "valve": "Point to every valve symbol (gate, ball, check, or control valve icon) in this P&ID tile.",
    "instrumentation": "Point to every instrument bubble or circle (gauge, transmitter, indicator) in this P&ID tile.",
    "pump": "Point to every pump symbol in this P&ID tile.",
    "tank": "Point to every tank or vessel symbol in this P&ID tile.",
    "inlet_outlet": "Point to every inlet/outlet or off-page connector symbol in this P&ID tile.",
    "general": ("Point to every other distinct P&ID equipment symbol in this tile not already a valve, "
                "instrument bubble, pump, tank, or inlet/outlet — e.g. flanges, nozzles, safety devices, reducers."),
}
assert set(MOLMO_PROMPTS_BY_TYPE) == set(MOLMO_ENTITY_TYPES)
print("Molmo will make", len(MOLMO_ENTITY_TYPES), "calls per tile:", MOLMO_ENTITY_TYPES)


## 7. PaddleOCR — stage 1.5 substitute (CPU)

Verbatim reuse of the Qwen-only Arm L notebook's `run_paddle_ocr` (PaddleOCR 3.7.0
`.predict()` shape). Runs on CPU — does not compete with Molmo/Qwen for VRAM. This
notebook needs OCR words for TWO things Arm P/the Qwen-only notebook didn't need them for
in quite the same way: (a) the normal `NormalizedWord` list `convert_detection` always
needs, and (b) the nearest-OCR-word pairing in §8 below, which is Molmo-specific.

In [ ]:
from pnid_agent.models.page_ocr import OcrWord

def run_paddle_ocr(png_path: str):
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(lang="en")
    result = ocr.predict(png_path)
    if not result:
        return []
    page = result[0]
    texts = page.get("rec_texts", [])
    scores = page.get("rec_scores", [])
    boxes = page.get("rec_boxes", [])
    words = []
    for text, score, box in zip(texts, scores, boxes):
        bbox = [int(round(v)) for v in box]
        words.append(OcrWord(text=text, bbox=bbox, confidence=float(score)))
    return words


## 8. Molmo -> NormalizedDetection, THEN nearest-OCR-word pairing (load-bearing fix)

**Why this pairing step exists — real, code-verified fact, not a guess** (verbatim from
`e2e_bench/types.py`'s `NormalizedDetection` docstring, itself confirmed by running the
real agent code):

> `value` is not cosmetic. The real `build_entity` (`stages/graph_construction/entities.py`)
> requires a non-empty derived name/tag to construct a `BundleEntity` at all —
> `if not derived_name or not source_bbox: return None, None, suggested`.
> `derived_name = detection.name or _derive_clean_label(detection)`, and
> `_derive_clean_label` needs grammar-reconstructed tags, Tag ID attributes, or a
> single-token raw `value` — with none of those, the entity is SILENTLY DROPPED before
> ever reaching stage 6/11/13/12, no error, no log the converter sees. **Molmo2's native
> output (points only, no text) therefore produces ZERO usable entities on its own.**

So: for every Molmo point, find the nearest PaddleOCR word (Euclidean distance, tile-local
PRE-upscale pixel space — i.e. divide the point's post-upscale tile coords by
`MOLMO_TILE_UPSCALE` before comparing, since `NormalizedDetection.bbox_tile` is POST-upscale
per `converters/stage04_detection.py`'s `TileBatch` docstring, but OCR words are only ever
in real, un-upscaled page pixels). If the nearest word is within `OCR_PAIR_RADIUS_PX`
(untuned guess, decision 4 — 120px chosen as a middle value of the 80-150px range this task
suggested; genuinely never validated against a real sheet in this pass), its text becomes
the detection's `value`. Detections with no OCR word within radius keep `value=None` and
are expected to be silently dropped downstream — that's the real, documented behavior
above, not a bug in this pairing step; the pairing-rate metric below reports how many
detections got paired vs. dropped so this isn't invisible.

In [ ]:
from e2e_bench.backends.parse_molmo import parse_molmo_points
from e2e_bench.types import NormalizedDetection

_n_paired = 0
_n_unpaired = 0


def molmo_detect_tile_all_types(tile_image, tile_w, tile_h) -> list:
    """Runs MOLMO_ENTITY_TYPES separate pointing calls on one tile crop (decision 1),
    returns the concatenated NormalizedDetection list (value=None — OCR pairing happens
    separately, see pair_with_ocr below, since it needs page-coord context this function
    doesn't have)."""
    dets = []
    for entity_type in MOLMO_ENTITY_TYPES:
        raw_text = molmo_generate(tile_image, MOLMO_PROMPTS_BY_TYPE[entity_type])
        outcome = parse_molmo_points(raw_text, tile_w, tile_h, entity_type=entity_type)
        if not outcome.parse_failed and outcome.value:
            dets.extend(outcome.value)
    return dets


def pair_with_ocr(detections: list, words_in_tile: list, tile_origin: tuple,
                   upscale: float, radius_px: float = OCR_PAIR_RADIUS_PX) -> list:
    """Mutates and returns `detections` (list[NormalizedDetection]) with `.value` set from
    the nearest OCR word within `radius_px`, tile-local PRE-upscale pixel space (see markdown
    above for why the /upscale conversion is needed). `words_in_tile`: NormalizedWord-like
    objects (here real `OcrWord`s from `slice_words_to_tile`) already sliced to this tile,
    still in PAGE coords — shifted by `tile_origin` to get tile-local."""
    global _n_paired, _n_unpaired
    tx0, ty0 = tile_origin
    word_centers = [((w.bbox[0] + w.bbox[2]) / 2.0 - tx0, (w.bbox[1] + w.bbox[3]) / 2.0 - ty0, w)
                    for w in words_in_tile]
    for det in detections:
        cx = (det.bbox_tile[0] + det.bbox_tile[2]) / 2.0 / upscale
        cy = (det.bbox_tile[1] + det.bbox_tile[3]) / 2.0 / upscale
        best_word, best_d = None, None
        for wx, wy, w in word_centers:
            d = ((wx - cx) ** 2 + (wy - cy) ** 2) ** 0.5
            if best_d is None or d < best_d:
                best_d, best_word = d, w
        if best_word is not None and best_d <= radius_px:
            # single-token value, per NormalizedDetection's docstring requirement
            token = best_word.text.split()[0] if best_word.text.split() else best_word.text
            det.value = token
            _n_paired += 1
        else:
            _n_unpaired += 1
    return detections


## 9. Real tiling + Molmo per-tile inference + OCR pairing

`compute_tile_grid` (real prod tiling: 1024px tiles, 205px overlap) and
`slice_words_to_tile` (real per-tile OCR-word slicing, `margin_px=24`, matching Arm P/the
Qwen-only Arm L notebook) run on CPU. `MOLMO_TILE_UPSCALE` is applied to the crop before
Molmo sees it (decision 2).

In [ ]:
from PIL import Image
from pnid_agent.sub_agents.symbol_detection.tile_words import slice_words_to_tile
from pnid_agent.stages.tile_segmentation.grid import compute_tile_grid

Image.MAX_IMAGE_PIXELS = None


def run_molmo_stage4(png_path: str, page_words: list):
    """Returns (per_tile_results: list[{tile_idx, origin_xy, detections}], W, H)."""
    global _n_paired, _n_unpaired
    _n_paired = _n_unpaired = 0

    full_img = Image.open(png_path).convert("RGB")
    W, H = full_img.size
    tiles = compute_tile_grid(drawing_bbox=[0, 0, W, H], page_size=(W, H))
    print(f"  {len(tiles)} tiles (1024/205 grid), upscale={MOLMO_TILE_UPSCALE}")

    per_tile_results = []
    for t in tiles:
        crop = full_img.crop((t.x0, t.y0, t.x1, t.y1))
        tile_w, tile_h = crop.size
        if MOLMO_TILE_UPSCALE != 1.0:
            crop = crop.resize((int(tile_w * MOLMO_TILE_UPSCALE), int(tile_h * MOLMO_TILE_UPSCALE)))
        upscaled_w, upscaled_h = crop.size

        dets = molmo_detect_tile_all_types(crop, upscaled_w, upscaled_h)
        words_in_tile = slice_words_to_tile(page_words, tile_bbox=[t.x0, t.y0, t.x1, t.y1], margin_px=24)
        dets = pair_with_ocr(dets, words_in_tile, (t.x0, t.y0), MOLMO_TILE_UPSCALE)

        n_with_value = sum(1 for d in dets if d.value)
        print(f"    tile {t.idx} ({t.x0},{t.y0})-({t.x1},{t.y1}): {len(words_in_tile)} ocr words, "
              f"{len(dets)} raw points, {n_with_value} paired to an OCR value")
        per_tile_results.append({"tile_idx": t.idx, "origin_xy": (t.x0, t.y0), "detections": dets})

    print(f"  OCR pairing: {_n_paired} paired / {_n_unpaired} unpaired "
          f"({_n_paired/(max(_n_paired+_n_unpaired,1)):.1%} paired) — unpaired detections "
          f"will be silently dropped by build_entity (see §8)")
    return per_tile_results, W, H


## 10. `LocalQwenMessagesClient`/`LocalQwenRunner` — real stage_13/12 drivers, local Qwen backend

Built in `src/e2e_bench/assembly/local_qwen_client.py`, mirroring
`real_openai_client.py`'s `RealOpenAIMessagesClient`/`RealOpenAIRunner` exactly (same
`.messages.create(...)` shape, same `content=[{"type":"tool_use",...}]` wrapping the real
drivers expect). Differences, all documented in that file's own docstring:

- **JSON-in-prompt, not native tool-calling** — the tool schema (`tools[0]["input_schema"]`)
  is rendered into an explicit fenced-```json``` instruction, schema-driven (works for
  stage 13's entity-verdict schema and stage 12's relation-verdict schema with the SAME
  renderer, no stage-specific branching), same technique the Qwen-only Arm L notebook
  already uses for stage 4.
- **Plain-text fallback** — `parse_json_common.py`'s own docstrings record that the real
  `v3-stage13`/`v3-relation` adapters were trained on plain "keep"/"remove" and "yes"/"no"
  answers, not JSON. If fenced-JSON extraction fails (or the parsed object is missing the
  tool's key field — `keep` or `verdict`), this client falls back to
  `parse_entity_verdict_json`/`parse_relation_verdict_json` verbatim. **Which path actually
  fires under the real model is untested** (intro decision 5).
- **Generic image-block handling** — an arbitrary number of `{"type":"image",...}` blocks
  decode to Qwen's `apply_chat_template` `{"type":"image","image": PIL.Image}` format, same
  requirement `real_openai_client.py` solved for OpenAI (stage 13 sends 1 image, stage 12
  sends 3 — no special-casing by count).
- **Adapter switching happens in the injected `generate_fn`, not the client itself** — the
  client's `adapter_name` field is metadata/debugging only (recorded per-call in
  `client.calls`); the actual `qwen_model.set_adapter(name)` call lives in this notebook's
  `make_qwen_generate_fn` (§11) so it fires immediately before every real `generate()` call,
  robust to stage 12's concurrent `asyncio.gather` dispatch (see the class docstring's
  concurrency note: `create()` awaits nothing internally, so concurrent callers naturally
  serialize on the one GPU model instance rather than truly overlapping).

Locally verified (no torch/GPU needed — `generate_fn` is injected, so this module has no
torch import at all): fenced-JSON success, plain-text keep/remove fallback, plain-text
yes/no fallback, a 3-image content list (stage 12's shape), and total-garbage-output ->
empty payload all round-trip correctly through `LocalQwenMessagesClient.messages.create`.

In [ ]:
from e2e_bench.assembly.local_qwen_client import LocalQwenMessagesClient, LocalQwenRunner

print("LocalQwenMessagesClient / LocalQwenRunner imported — see src/e2e_bench/assembly/local_qwen_client.py")


## 11. Free Molmo, load Qwen3-VL-8B + BOTH adapters (multi-adapter, one base model)

Same `peft` multi-adapter pattern as
`Stage105_SkidMatrix_Molmo2_Qwen_Adapters_GPUOnly.ipynb` section 6: attach `v3-stage13` via
`PeftModel.from_pretrained(base, path, adapter_name="stage13")`, then attach `v3-relation`
via `.load_adapter(path, adapter_name="relation")` on the SAME `PeftModel` instance — one
base model in VRAM, not two, `set_adapter(name)` switches which LoRA is active before each
generate call. This is also where cascade mode's one GPU-model handoff happens — Molmo is
freed here and never reloaded later in this notebook (see §15's RUN_ALL_4 caveat).

In [ ]:
del molmo_model, molmo_processor
torch.cuda.empty_cache()
print("Molmo2 freed. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

from peft import PeftModel
from huggingface_hub import snapshot_download

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_base = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda").eval()
print("Qwen3-VL-8B base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def _pull(adapter_path):
    local = Path(f"/content/adp_{adapter_path.replace('/', '_')}")
    snapshot_download(repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
                       allow_patterns=[f"{adapter_path}/*"], local_dir=str(local))
    d = local / adapter_path
    assert (d / "adapter_model.safetensors").exists(), f"missing: {d}"
    return str(d)

adapter_names = list(ADAPTERS)   # ["stage13", "relation"]
first = adapter_names[0]
qwen_model = PeftModel.from_pretrained(qwen_base, _pull(ADAPTERS[first]), adapter_name=first)
for name in adapter_names[1:]:
    qwen_model.load_adapter(_pull(ADAPTERS[name]), adapter_name=name)
qwen_model.eval()
print(f"adapters attached: {adapter_names}")


def make_qwen_generate_fn(adapter_name: str):
    """generate_fn injected into LocalQwenMessagesClient — sets the active adapter
    immediately before every real generate() call (see §10 markdown for why this, not the
    client, owns the actual peft switch)."""
    def _generate(qwen_content: list, max_new_tokens: int) -> str:
        qwen_model.set_adapter(adapter_name)
        messages = [{"role": "user", "content": qwen_content}]
        inputs = qwen_processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt").to(qwen_model.device)
        with torch.no_grad():
            out = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[0][inputs["input_ids"].shape[1]:]
        return qwen_processor.decode(gen, skip_special_tokens=True).strip()
    return _generate


runner13 = LocalQwenRunner(LocalQwenMessagesClient(
    make_qwen_generate_fn("stage13"), adapter_name="stage13", max_new_tokens=QWEN_MAX_NEW_TOKENS))
runner12 = LocalQwenRunner(LocalQwenMessagesClient(
    make_qwen_generate_fn("relation"), adapter_name="relation", max_new_tokens=QWEN_MAX_NEW_TOKENS))
print("runner13 (v3-stage13) and runner12 (v3-relation) ready")


## 12. Real production schemas (for stage_13_run's `schema_factory` bypass)

Same `build_benchmark_schemas()` this project's Qwen-only Arm L notebook and Arm P v3 both
use — real `EntityExtractionSchema` objects, empty attribute model (`_EmptyAttrs`, matching
this benchmark's ontology having no real per-type attributes), bypasses the tenant-ontology
fetch so `stage_13_run` uses the SAME benchmark entity types everywhere else in this
notebook.

In [ ]:
from pydantic import BaseModel

from e2e_bench.ontology import load_benchmark_ontology_raw
from pnid_agent.sub_agents.title_block_extraction.ontology import EntityExtractionSchema


class _EmptyAttrs(BaseModel):
    pass


def build_benchmark_schemas():
    return [
        EntityExtractionSchema(entity_type=k, model=_EmptyAttrs, attribute_metadata={},
                                raw_sample_payload={"descriptions": {"en": v}})
        for k, v in entity_type_names().items()
    ]


schemas = build_benchmark_schemas()
print(f"{len(schemas)} benchmark entity-type schemas built")


## 13. Full cascade: Molmo Stage 4 -> real Stage 6 -> entities/relations -> checkpoint 1
## -> real Stage 13 (Qwen+v3-stage13) -> checkpoint 2 -> real Stage 12 (Qwen+v3-relation) -> checkpoint 3

Identical structure to Arm P v3 (`poc_run_arm_p_v3.py`) from `build_relations` onward:
pack into a real `RiveOntology`, write `stage-11/rive_ontology.json`, call `stage_13_run`
then `stage_12_run` unmodified with the local runners from §11, re-score at each
checkpoint using `contract_to_equipment_edges` (NOT PID2Graph's raw edges — see the intro's
"Proven, reused verbatim" list for why).

In [ ]:
import json
import tempfile
from types import SimpleNamespace

from e2e_bench.assembly.document import build_artifact_store, build_single_page_document
from e2e_bench.assembly.entities import detections_to_entities
from e2e_bench.converters.stage01_classification import convert_classification
from e2e_bench.converters.stage04_detection import TileBatch, convert_detection
from e2e_bench.ontology import load_ontology_relation_index
from e2e_bench.types import NormalizedWord

from e2e_harness.graph_matcher import match_entities, match_relations
from e2e_harness.ground_truth import contract_to_equipment_edges, equipment_only, parse_graphml_ground_truth

from pnid_agent.models.page_classification import PageClassificationLabel
from pnid_agent.models.line_tracing import Stage06Output
from pnid_agent.models.rive_ontology import DrawingMetadata, RiveOntology
from pnid_agent.stages.graph_construction.relations import build_relations
from pnid_agent.stages.line_tracing.driver import stage_06_run
from pnid_agent.sub_agents.entity_validation.driver import stage_13_run
from pnid_agent.sub_agents.relation_validation.driver import stage_12_run


def score(entities, relations, gt_equip, gt_edges_contracted, label: str) -> dict:
    em = match_entities(entities, gt_equip)
    rm = match_relations(relations, gt_edges_contracted, em)
    result = {
        "checkpoint": label, "n_entities": len(entities), "n_relations": len(relations),
        "entity_precision": em.precision, "entity_recall": em.recall, "entity_f1": em.f1,
        "relation_precision": rm.precision, "relation_recall": rm.recall, "relation_f1": rm.f1,
    }
    print(f"  [{label}] entity P={em.precision:.3f} R={em.recall:.3f} F1={em.f1:.3f} | "
          f"relation P={rm.precision:.3f} R={rm.recall:.3f} F1={rm.f1:.3f}")
    return result


def rive_ontology_from_bundle(entities, relations) -> RiveOntology:
    return RiveOntology(ontology_version="benchmark-v1", drawing=DrawingMetadata(),
                         entities=entities, relations=relations)


async def run_arm_l_mixed(sheet_id: str, graphml_path: str, png_path: str) -> dict:
    print(f"=== Arm L mixed (Molmo2-O-7B stage4, Qwen3-VL-8B+adapters stage13/12) on {sheet_id} ===")

    store = build_artifact_store(tempfile.mkdtemp())
    doc = build_single_page_document(
        doc_id=sheet_id, job_id="job-armLmixed-" + sheet_id, tenant_id="benchmark",
        image_path=png_path, artifact_store=store,
    )
    context = SimpleNamespace(tenant_id="benchmark")

    convert_classification(
        drawing_document=doc, artifact_store=store, page_index=0,
        classification=PageClassificationLabel.PID_DRAWING, confidence=1.0,
        model_version="molmo2-o-7b-assumed",
    )

    print("  running PaddleOCR (stage 1.5 substitute)...")
    page_words = run_paddle_ocr(png_path)
    print(f"  {len(page_words)} OCR words")

    print("  running Molmo2 stage 4 (per-tile, per-type pointing + OCR pairing)...")
    per_tile_results, W, H = run_molmo_stage4(png_path, page_words)

    tile_batches = [
        TileBatch(tile_index=r["tile_idx"], origin_xy=r["origin_xy"],
                  upscale=MOLMO_TILE_UPSCALE, detections=r["detections"])
        for r in per_tile_results
    ]
    normalized_ocr_words = [NormalizedWord(text=w.text, bbox=w.bbox, confidence=w.confidence) for w in page_words]

    s4_out, dropped = convert_detection(
        drawing_document=doc, artifact_store=store, page_index=0,
        tile_batches=tile_batches, ocr_words_for_page=normalized_ocr_words,
        model_version="molmo2-o-7b",
    )
    print(f"  stage4: {len(s4_out.pages[0].detections)} detections after NMS, {len(dropped)} dropped in compose")

    await stage_06_run(context, store, drawing_document=doc)
    s6_raw = store.read_json(doc.job_id, "stage-06/stage_06_output.json")
    s6_out = Stage06Output.model_validate(s6_raw)
    print(f"  stage6: {len(s6_out.pages[0].segments)} segments")

    entities, det_to_temp, type_by_temp = detections_to_entities(
        detections=s4_out.pages[0].detections, page_index=0, page_size=(W, H),
        stage_4_model_version="molmo2-o-7b",
    )
    print(f"  entities built: {len(entities)} (of {len(s4_out.pages[0].detections)} detections)")

    ontology_idx = load_ontology_relation_index()
    relations, _meta, unresolved = build_relations(s6_out.pages[0], det_to_temp, type_by_temp, ontology_idx)
    print(f"  relations built: {len(relations)}, unresolved: {len(unresolved)}")

    gt_entities, gt_edges_raw = parse_graphml_ground_truth(graphml_path)
    gt_equip = equipment_only(gt_entities)
    gt_edges_contracted = contract_to_equipment_edges(gt_entities, gt_edges_raw)
    print(f"  GT: {len(gt_equip)} equipment entities, {len(gt_edges_raw)} raw edges -> "
          f"{len(gt_edges_contracted)} contracted equipment edges")

    checkpoints = {}
    checkpoints["pre_13_12"] = score(entities, relations, gt_equip, gt_edges_contracted, "pre-13/12")

    rive = rive_ontology_from_bundle(entities, relations)
    store.write_json(doc.job_id, "stage-11/rive_ontology.json", rive.ui_payload())
    print(f"  wrote stage-11/rive_ontology.json: {len(rive.entities)} entities, {len(rive.relations)} relations")

    print("  running REAL stage_13_run (entity validation, Qwen3-VL-8B + v3-stage13)...")
    await stage_13_run(
        context, store, drawing_document=doc,
        schema_factory=lambda: schemas,
        vlm_runner=runner13,
        model="qwen3vl-8b+v3-stage13",
    )
    s13_payload = store.read_json(doc.job_id, "stage-13/rive_ontology.json")
    rive_13 = RiveOntology.model_validate(s13_payload)
    print(f"  stage13 done: {len(rive_13.entities)} entities, {len(rive_13.relations)} relations after validation")
    checkpoints["post_13"] = score(rive_13.entities, rive_13.relations, gt_equip, gt_edges_contracted, "post-13")

    print("  running REAL stage_12_run (relation validation, Qwen3-VL-8B + v3-relation)...")
    raw_ontology = load_benchmark_ontology_raw()
    await stage_12_run(
        context, store, drawing_document=doc,
        ontology_payload_factory=lambda: {"relations": raw_ontology["relations"]},
        vlm_runner=runner12,
        model="qwen3vl-8b+v3-relation",
        source_rive_uri="stage-13/rive_ontology.json",
    )
    s12_payload = store.read_json(doc.job_id, "stage-12/rive_ontology.json")
    rive_12 = RiveOntology.model_validate(s12_payload)
    print(f"  stage12 done: {len(rive_12.entities)} entities, {len(rive_12.relations)} relations after validation")
    checkpoints["post_12"] = score(rive_12.entities, rive_12.relations, gt_equip, gt_edges_contracted, "post-12")

    result = {
        "sheet_id": sheet_id, "arm": "molmo2o7b-stage4_qwen3vl8b-adapters-stage13-stage12",
        "n_gt_entities": len(gt_equip), "n_gt_edges_contracted": len(gt_edges_contracted),
        "ocr_pairing": {"n_paired": _n_paired, "n_unpaired": _n_unpaired},
        "checkpoints": checkpoints,
    }
    print(json.dumps(result, indent=2))
    return result


### Async note

Same as the Qwen-only Arm L notebook: Colab's IPython kernel already runs an event loop, so
`asyncio.run(...)` doesn't work unmodified in a notebook cell — `nest_asyncio` patches
this.

In [ ]:
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()


## 14. Run on the primary holdout sheet

Compare `checkpoints.pre_13_12/post_13/post_12` directly against Arm P v3's row
(`gpt-5.5-low-v3-realprompt-paddleocr-stage13-stage12`) and the Qwen-only Arm L notebook's
number, on this SAME sheet.

In [ ]:
result = await run_arm_l_mixed(sheet["sheet_id"], sheet["graphml_path"], sheet["png_path"])


## 15. Optional — all 4 frozen holdout sheets

`src/e2e_harness/e2e_holdout_ids.json` freezes 4 sheets. **Caveat: Molmo was already freed
in §11 by the time you reach this cell** — `run_arm_l_mixed` calls `run_molmo_stage4`, which
uses the module-global `molmo_model`/`molmo_processor`; if those were deleted (they always
are, per this notebook's cascade design), running `RUN_ALL_4` requires re-running §5's Molmo
load cell (and re-freeing + reloading Qwen adapters after, per §11) for EACH additional
sheet's Stage 4 pass — not automated here, flagged rather than silently failing on a
`NameError`.

In [ ]:
import e2e_harness.holdout as holdout_mod

RUN_ALL_4 = False   # flip to True only after re-reading the Molmo-reload caveat above

if RUN_ALL_4:
    all_results = []
    for s in holdout_mod.load_holdout():
        sh = resolve_sheet(s["tree"], s["stem"])
        r = await run_arm_l_mixed(sh["sheet_id"], sh["graphml_path"], sh["png_path"])
        all_results.append(r)
    print(json.dumps(all_results, indent=2))


## 16. Push results to HF + free the GPU

Same "no MLflow, append to a flat results artifact" convention this project uses
everywhere. Copy the printed `entity_*`/`relation_*` numbers into `pid-ml/results.csv` by
hand afterward (per `CLAUDE.md`'s schema), same as every other run in this repo.

In [ ]:
from huggingface_hub import HfApi

with open("/content/arm_l_mixed_result.json", "w") as f:
    json.dump({"primary_sheet": result, **({"all_4": all_results} if RUN_ALL_4 else {})}, f, indent=2)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/arm_l_mixed_result.json",
    path_in_repo=f"benchmarks/arm_l_molmo2_qwen_mixed_{sheet['sheet_id']}.json",
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print("results pushed to HF")

from google.colab import runtime
runtime.unassign()
